# MolecularFunction → ChemicalEntity Relation Pipeline

Builds a unified, deduplicated edge table for the strictly **MolecularFunction–ChemicalEntity** relation (no swapping, only native MolecularFunction → ChemicalEntity).

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch (Human_MolecularActivity_ChemicalEntity_Monarch - Native)
- PrimeKG (PrimeKG_MolecularFunction_ChemicalEntity1 - Native)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MOLECULARFUNCTION_CHEMICALENTITY/ALL_MOLECULARFUNCTION_CHEMICALENTITY.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

## 1 · Mapping DBs

In [3]:
# GO Dictionary mapping
GO = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(GO['id'], GO['name']))
del GO


Drugbank = pd.read_csv(MAPPING_DIR + 'drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))
del Drugbank


## 2 · Load Source Files

In [4]:
# 1. Monarch (MolecularActivity -> ChemicalEntity: Native)
Monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_MolecularActivity_ChemicalEntity_Monarch.csv')
Monarch.columns = Monarch.columns.str.lower()
if 'kgsource' in Monarch.columns:
    Monarch = Monarch.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'tail_name': 'tail_detail_name'})
required_columns = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]
Monarch = Monarch[required_columns]
Monarch['head_id_is'] = 'GO'
Monarch['relation'] = 'MolecularFunction_ChemicalEntity'
Monarch['head_type'] = 'MolecularFunction'
Monarch['tail_type'] = 'ChemicalEntity'
Monarch['kg_type'] = 'Generalised'
print(f"Monarch: {len(Monarch):,} rows")

Monarch

Monarch: 17,648 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type
0,GO:0061459,MolecularFunction_ChemicalEntity,1549073,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,L-arginine transmembrane transporter activity,L-argininium(1+),Generalised
1,GO:0061501,MolecularFunction_ChemicalEntity,5461108,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",ATP(4-),Generalised
2,GO:0061501,MolecularFunction_ChemicalEntity,4097574,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",diphosphate(3-),Generalised
3,GO:0061501,MolecularFunction_ChemicalEntity,135398632,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",GTP(4-),Generalised
4,GO:0061513,MolecularFunction_ChemicalEntity,3681305,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,glucose 6-phosphate:phosphate antiporter activity,hydrogenphosphate,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...
17643,GO:0055041,MolecularFunction_ChemicalEntity,1038,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,cyclopentanol dehydrogenase activity,hydron,Generalised
17644,GO:0055041,MolecularFunction_ChemicalEntity,7298,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,cyclopentanol dehydrogenase activity,cyclopentanol,Generalised
17645,GO:0055041,MolecularFunction_ChemicalEntity,8452,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,cyclopentanol dehydrogenase activity,cyclopentanone,Generalised
17646,GO:0055041,MolecularFunction_ChemicalEntity,15938971,MolecularFunction,related_to,ChemicalEntity,MonarchKG,GO,Pubchem,cyclopentanol dehydrogenase activity,NAD(1-),Generalised


In [5]:
# 2. PrimeKG (MolecularFunction -> ChemicalEntity: Native)
PrimeKG = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_MolecularFunction_ChemicalEntity1.csv')
PrimeKG.columns = PrimeKG.columns.str.lower()
PrimeKG['tail'] = PrimeKG['tail'].astype(str).str.replace(r'\.0$', '', regex=True)
PrimeKG = PrimeKG[PrimeKG['tail'] != 'nan']
if 'kgsource' in PrimeKG.columns:
    PrimeKG = PrimeKG.rename(columns={'kgsource': 'kg_source'})
if 'head_go_id' in PrimeKG.columns:
    PrimeKG = PrimeKG.rename(columns={'head_go_id': 'head_detail_name'})
if 'tail_name' in PrimeKG.columns:
    PrimeKG = PrimeKG.rename(columns={'tail_name': 'tail_detail_name'})
required_columns = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]
PrimeKG = PrimeKG[required_columns]
PrimeKG['head_id_is'] = 'GO'
PrimeKG['relation'] = 'MolecularFunction_ChemicalEntity'
PrimeKG['head_type'] = 'MolecularFunction'
PrimeKG['tail_type'] = 'ChemicalEntity'
PrimeKG['kg_source'] = 'PrimeKG'
PrimeKG['kg_type'] = 'Generalised'
print(f"PrimeKG: {len(PrimeKG):,} rows")
PrimeKG

PrimeKG: 21 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type
0,GO:0019766,MolecularFunction_ChemicalEntity,37034,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,IgA receptor activity,"1,2,4-trichloro-5-(2,4,5-trichlorophenyl)benzene",Generalised
1,GO:0019766,MolecularFunction_ChemicalEntity,98491,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,IgA receptor activity,1-chloro-4-[2-chloro-1-(4-chlorophenyl)ethenyl...,Generalised
2,GO:0019770,MolecularFunction_ChemicalEntity,37034,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,IgG receptor activity,"1,2,4-trichloro-5-(2,4,5-trichlorophenyl)benzene",Generalised
3,GO:0019770,MolecularFunction_ChemicalEntity,98491,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,IgG receptor activity,1-chloro-4-[2-chloro-1-(4-chlorophenyl)ethenyl...,Generalised
4,GO:0004104,MolecularFunction_ChemicalEntity,9920327,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,"(1'R,2R,3S,4'S,6S,8'R,10'E,12'S,13'S,14'E,16'E...",Generalised
5,GO:0004104,MolecularFunction_ChemicalEntity,5359596,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,arsenic,Generalised
6,GO:0004104,MolecularFunction_ChemicalEntity,5463849,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,dicopper;dichloride;trihydroxide,Generalised
7,GO:0004104,MolecularFunction_ChemicalEntity,3082,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,2-dimethoxyphosphinothioylsulfanyl-N-methylace...,Generalised
10,GO:0004104,MolecularFunction_ChemicalEntity,4130,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,dimethoxy-(4-nitrophenoxy)-sulfanylidene-lambd...,Generalised
13,GO:0004104,MolecularFunction_ChemicalEntity,123047,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,GO,Pubchem,cholinesterase activity,triazine,Generalised


## 3 · Consolidate & Map ID Types

In [10]:
source_dfs = [Monarch, PrimeKG]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']
print(final_df.isna().sum())

final_df

head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,GO:0061459,MolecularFunction_ChemicalEntity,1549073,MolecularFunction,related_to,ChemicalEntity,MonarchKG,Generalised,GO,Pubchem,L-arginine transmembrane transporter activity,L-argininium(1+)
1,GO:0061501,MolecularFunction_ChemicalEntity,5461108,MolecularFunction,related_to,ChemicalEntity,MonarchKG,Generalised,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",ATP(4-)
2,GO:0061501,MolecularFunction_ChemicalEntity,4097574,MolecularFunction,related_to,ChemicalEntity,MonarchKG,Generalised,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",diphosphate(3-)
3,GO:0061501,MolecularFunction_ChemicalEntity,135398632,MolecularFunction,related_to,ChemicalEntity,MonarchKG,Generalised,GO,Pubchem,"2',3'-cyclic GMP-AMP synthase activity",GTP(4-)
4,GO:0061513,MolecularFunction_ChemicalEntity,3681305,MolecularFunction,related_to,ChemicalEntity,MonarchKG,Generalised,GO,Pubchem,glucose 6-phosphate:phosphate antiporter activity,hydrogenphosphate
...,...,...,...,...,...,...,...,...,...,...,...,...
17664,GO:0004021,MolecularFunction_ChemicalEntity,23667657,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,Generalised,GO,Pubchem,L-alanine:2-oxoglutarate aminotransferase acti...,"sodium;2,2,3,3,4,4,5,5,6,6,7,7,8,8,8-pentadeca..."
17665,GO:0022853,MolecularFunction_ChemicalEntity,133082519,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,Generalised,GO,Pubchem,active ion transmembrane transporter activity,hexapotassium;hexasodium;3-carboxy-3-hydroxype...
17666,GO:0022853,MolecularFunction_ChemicalEntity,753,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,Generalised,GO,Pubchem,active ion transmembrane transporter activity,"propane-1,2,3-triol"
17667,GO:0022853,MolecularFunction_ChemicalEntity,11957628,MolecularFunction,interacts with,ChemicalEntity,PrimeKG,Generalised,GO,Pubchem,active ion transmembrane transporter activity,"(2R,3R)-2,3-dihydroxybutanedioic acid;3-[(2S)-..."


## 4 · GO & Chemical Name Normalisation & NA Audit Before Deduplication

In [11]:
# Fill detail names using MESH GO dictionary (for Head MolecularFunction)
head_detail = final_df['head'].map(GO_dict)
final_df['head_detail_name'] = final_df['head_detail_name'].fillna(head_detail)

# Fill detail names using Drugbank mapping dictionary
if Drugbank_dict:
    tail_detail = final_df['tail'].map(Drugbank_dict)
    final_df['tail_detail_name'] = final_df['tail_detail_name'].fillna(tail_detail)


print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 17,669

NaN audit before deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


## 5 · Deduplication & NA Audit After Deduplication

In [12]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))
group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first', 'relation_type': 'first', 'tail_type': 'first',
    'kg_source': merge_sources, 'kg_type': merge_sources,
    'head_id_is': 'first', 'tail_id_is': 'first',
    'head_detail_name': 'first', 'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
print("\nNaN audit after deduplication:")
print(final_df_dedup.isna().sum())

Before dedup: 17,669  |  After dedup: 17,669

NaN audit after deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


## 6 · Save Output

In [13]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 17,669 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MOLECULARFUNCTION_CHEMICALENTITY/ALL_MOLECULARFUNCTION_CHEMICALENTITY.csv
